In [10]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
og_df = pd.read_csv('regional_stats.csv').iloc[:, 1:]
og_df

,country,state,lpopd,egood,ebad,mining,plantations,enone,yppp,lyppp,...,lseats,native,black,temp_avg,temp2,rainfall,rain2,alti,alti2,landlocked
0,Argentina,Buenos Aires,-2.974981,1,0,0.0,0.0,0,9321.192383,9.140046,...,-12.095780,2.508355,NaN,15.875000,252.01562,1.0926,1.193775,0.026,0.000676,0.0
1,Argentina,Catamarca,-1.790302,1,0,0.0,0.0,0,7304.713379,8.896275,...,-10.877821,2.522629,NaN,20.525000,421.27560,0.4581,0.209856,0.519,0.269361,1.0
2,Argentina,Chaco,-0.462096,1,0,0.0,0.0,0,5624.872070,8.634953,...,-11.645584,3.627478,NaN,21.008333,441.35007,1.5567,2.423315,0.047,0.002209,1.0
3,Argentina,Chubut,-4.711761,0,0,0.0,0.0,1,13967.312500,9.544475,...,-11.080855,9.685770,NaN,13.050000,170.30250,0.2283,0.052121,0.003,0.000009,0.0
4,Argentina,Ciudad de Buenos Aires (Capital Federal),-2.974981,1,0,0.0,0.0,0,30949.996094,10.340128,...,-11.693580,2.316357,NaN,17.725000,314.17563,1.2146,1.475253,0.010,0.000100,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340,Venezuela,Portuguesa,0.367317,1,0,0.0,0.0,0,3496.789062,8.159600,...,-11.498904,0.200000,NaN,26.500000,702.25000,1.6050,2.576025,0.172,0.029584,1.0
341,Venezuela,Sucre,1.023031,0,0,0.0,0.0,1,4276.005859,8.360775,...,-11.417274,1.000000,NaN,26.850000,720.92255,0.7280,0.529984,0.060,0.003600,0.0
342,Venezuela,Trujillo,0.576342,1,0,0.0,0.0,0,4283.611328,8.362552,...,-11.468277,0.200000,NaN,25.100000,630.01000,1.1870,1.408969,0.884,0.781456,1.0
343,Venezuela,Táchira,0.431626,1,0,0.0,0.0,0,5822.441895,8.669475,...,-11.527893,0.200000,NaN,24.866667,618.35114,1.4970,2.241009,1.213,1.471369,1.0


In [24]:
""" Replicate Table 1. Regional PPP GDP per Capita across the Americas"""
yppp_stat_region = og_df.groupby("country").agg(
    Observations=("yppp", "count"),
    Mean=("yppp", "mean"),
    Maximum=("yppp", "max"), 
    Minimum=("yppp", "min")
)
yppp_stat_region["Log S.D."] = og_df.groupby("country")["lyppp"].std()
yppp_stat_region["Ratio y max / y min"] = yppp_stat_region["Maximum"] / yppp_stat_region["Minimum"]

# rearranging and rounding to match the og table
yppp_stat_region = yppp_stat_region[['Observations', 'Mean', 'Log S.D.', 'Minimum', 'Maximum', 'Ratio y max / y min']]
yppp_stat_region = yppp_stat_region.round({'Log S.D.': 3, 'Ratio y max / y min': 2})
int_cols = ['Observations', 'Mean', 'Minimum', 'Maximum']
yppp_stat_region[int_cols] = yppp_stat_region[int_cols].astype(int)
yppp_stat_region

,Observations,Mean,Log S.D.,Minimum,Maximum,Ratio y max / y min
country,,,,,,
Argentina,24,11705,0.553,4578,40450,8.84
Bolivia,9,2715,0.395,1244,4222,3.39
Brazil,27,5753,0.576,1792,17595,9.81
Canada,13,44267,0.358,26941,94900,3.52
Chile,13,8728,0.423,4154,19820,4.77
Colombia,30,5868,0.489,2367,22314,9.43
Ecuador,22,5057,0.834,1457,26573,18.23
El Salvador,12,3336,0.300,2191,5954,2.72
Guatemala,8,3562,0.439,2100,8400,4.00


In [26]:
yppp_stat_region.to_latex(
    "table1_regional_gdp.tex",
    index=True,
    caption="Regional PPP GDP per Capita across the Americas",
    label="tab:regional_gdp"
)

In [25]:
""" Replicate Table 2. Summary Statistics"""
t2_cols = [
    'lyppp', 'lpoverty', 'lhealth', 'lgini', 'lschoolspk', 'llit', 'lseats',
    'egood', 'ebad', 'mining', 'plantations', 'enone', 'lpopd',
    'temp_avg', 'temp2', 'rainfall', 'rain2', 'alti', 'alti2', 'landlocked'
]
new_var_map = {
    'lyppp': 'Log PPP GDP per capita',
    'lpoverty': 'Log poverty rate',
    'lhealth': 'Health Index',
    'lgini': 'Log Gini',
    'lschoolspk': 'Log schools per child',
    'llit': 'Log literacy rate',
    'lseats': 'Log seats in lower house per voter',
    'egood': 'Good activities dummy',
    'ebad': 'Bad activities dummy',
    'mining': 'Mining dummy',
    'plantations': 'Plantations dummy',
    'enone': 'No activities dummy',
    'lpopd': 'Log precolonial population density',
    'temp_avg': 'Average temperature',
    'temp2': 'Average temperature squared',
    'rainfall': 'Total rainfall',
    'rain2': 'Total rainfall squared',
    'alti': 'Altitude',
    'alti2': 'Altitude squared',
    'landlocked': 'Landlocked dummy'
}
new_stat_map = {
    'count': 'Observations',
    'mean': 'Mean',
    'std': 'S.D.',
    'min': 'Minimum',
    'max': 'Maximum'
}

# generate stats, rename columns and rows
summary_stats = og_df[t2_cols].agg(['count', 'mean', 'std', 'min', 'max'])
summary_stats = summary_stats.rename(columns=new_var_map, index=new_stat_map)

# transpose and round to match og table
summary_stats = summary_stats.T
summary_stats = summary_stats.round({'Mean' : 2, 'S.D.' : 2, 'Minimum' : 2, 'Maximum' : 2})
summary_stats['Observations'] = summary_stats['Observations'].astype(int)
summary_stats.index.name = 'Outcome Variables'
summary_stats

,Observations,Mean,S.D.,Minimum,Maximum
Outcome Variables,,,,,
Log PPP GDP per capita,345,8.83,0.97,7.13,11.67
Log poverty rate,331,2.93,0.92,0.21,4.40
Health Index,53,4.22,0.38,2.95,4.52
Log Gini,268,-0.74,0.16,-1.15,-0.46
Log schools per child,317,-5.31,0.64,-7.29,-3.69
Log literacy rate,270,-0.14,0.13,-0.76,-0.00
Log seats in lower house per voter,318,-11.42,1.19,-13.59,-8.14
Good activities dummy,345,0.61,0.49,0.00,1.00
Bad activities dummy,345,0.21,0.41,0.00,1.00


In [27]:
summary_stats.to_latex(
    "table2_summary_stats.tex",
    index=True,
    caption="Summary Statistics",
    label="tab:summary_stats"
)

In [32]:
""" Table 4 Panel A"""

# define independant variables for regression
colo_regr_df = og_df.copy()
X_cols = ['lpopd', 'temp_avg', 'temp2', 'rainfall', 'rain2', 'alti', 'alti2', 'landlocked']
X = sm.add_constant(colo_regr_df[X_cols])

# create colonial activity columns
colo_regr_df["bad"]  = ((colo_regr_df["mining"] == 1) |
                        (colo_regr_df["plantations"] == 1)).astype(int)
colo_regr_df["none"] = colo_regr_df["enone"]

pop_cut = colo_regr_df["lpopd"].median()
colo_regr_df["good"] = ((colo_regr_df["bad"] == 0) &
                        (colo_regr_df["none"] == 0) &
                        (colo_regr_df["lpopd"] <= pop_cut)).astype(int)
colo_regr_df["ugly"] = ((colo_regr_df["bad"] == 0) &
                        (colo_regr_df["none"] == 0) &
                        (colo_regr_df["lpopd"]  > pop_cut)).astype(int)

y_vars = {
    "bad": "Bad Activities",
    "mining": "Mining",
    "plantations": "Plantations",
    "good": "Good Activities",
    "ugly": "Ugly Activities",
    "none": "No Activities"
}

# run regression
panel_a_results = {}
for col, label in y_vars.items():
    y = colo_regr_df[col]
    panel_a_results[label] = sm.OLS(y, X).fit()
with open("panel_a_regression_output.txt", "w") as f:
    for label, result in panel_a_results.items():
        f.write(f"\n===== Regression for: {label} =====\n")
        f.write(result.summary().as_text())
        f.write("\n\n")

In [33]:
""" Table 4 Panel B"""

# define independant variables, add country dummies
base_X = ['lpopd','temp_avg','temp2','rainfall','rain2','alti','alti2','landlocked']
country_dummies = pd.get_dummies(colo_regr_df['country'], drop_first=True, dtype=float)
X = sm.add_constant(pd.concat([colo_regr_df[base_X], country_dummies], axis=1)).astype(float)

# run regression
panel_b_results = {}
for col, label in y_vars.items():
    y = colo_regr_df[col].astype(float)
    panel_b_results[label] = sm.OLS(y, X).fit()

with open("panel_b_regression_output.txt", "w") as f:
    for label, result in panel_b_results.items():
        f.write(f"\n===== Panel B Regression for: {label} =====\n")
        f.write(result.summary().as_text())
        f.write("\n\n")


In [44]:
""" generate tables to match the paper"""

def star_sig(p):
    """return significance stars."""
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''

def generate_panel(results):
    # columns and rows 
    col_order = ["Bad Activities", "Mining", "Plantations", "Good Activities", "Ugly Activities", "No Activities"]
    var_order = [
        ('lpopd',     "Log precolonial population density"),    # rename
        ('temp_avg',  "Average temperature"),
        ('temp2',     "Average temperature squared"),
        ('rainfall',  "Total rainfall"),
        ('rain2',     "Total rainfall squared"),
        ('alti',      "Altitude"),
        ('alti2',     "Altitude squared"),
        ('landlocked',"Landlocked dummy")
    ]
    
    output_df = pd.DataFrame(index=[ind for _, ind in var_order],
                   columns=col_order, dtype=str)

    for out in col_order:
        res = results[out]                 # statsmodels result
        for var, pretty in var_order:
            coef  = res.params[var]
            se    = res.bse[var]
            cell  = f"{coef:.3f}{star_sig(res.pvalues[var])}\n({se:.3f})"
            output_df.loc[pretty, out] = cell

    # add R_2 row
    r2_row = {out: f"{results[out].rsquared:.3f}" for out in col_order}
    output_df.loc["$R^2$"] = r2_row

    return output_df

In [47]:
generate_panel(panel_a_results).to_latex("table4_panelA.tex", escape=False)
generate_panel(panel_b_results).to_latex("table4_panelB.tex", escape=False)

In [105]:
def tbl_56_regr(df, groups, regressors, y, country_dummies):
    """ helper function to do the regression for table 5 and 6"""
    tbl = {} # dictionary {column‑label : fitted result}
    for col, X_base in regressors.items():
        # make sure to include country dummies
        X = sm.add_constant(pd.concat([df[X_base], country_dummies], axis=1))
        res = sm.OLS(y, X).fit(
                cov_type = 'cluster',
                cov_kwds = {'groups': groups}
            )
        tbl[col] = res
        
    return tbl

def coef_cell(res, par):
    """Return a string like  -0.392***\n(0.099)  for parameter <par>."""
    if par not in res.params:
        return ""
    b  = res.params[par]
    se = res.bse[par]
    # stars ---------------------------------------------------------
    pval  = res.pvalues[par]
    stars = "***" if pval < .01 else ("**" if pval < .05 else ("*" if pval < .10 else ""))
    return f"{b:.3f}{stars}\n({se:.3f})"



In [106]:
""" Replicate Table 5. Colonial Activities and Current GDP per Capita"""
df = colo_regr_df.copy()
y  = df['lyppp']
groups = df['lpopd']
country_dummies = pd.get_dummies(df['country'], drop_first=True, dtype=float)

regressors = {
    '(1)': ['good', 'bad', 'ugly'],                                            # only colonial‑activity dummies
    '(2)': ['lpopd'],                                                          # only log pre‑colonial pop. density
    '(3)': ['good', 'bad', 'ugly', 'lpopd'],                                   # add pop density
    '(4)': ['good', 'bad', 'ugly', 'lpopd',                                    # + climate controls
            'temp_avg', 'temp2', 'rainfall', 'rain2'],
    '(5)': ['good', 'bad', 'ugly', 'lpopd',                                    # + geography controls
            'temp_avg', 'temp2', 'rainfall', 'rain2',
            'alti', 'alti2', 'landlocked'],
    '(6)': ['good', 'ugly', 'plantations', 'mining', 'lpopd',                 # split “bad” into two dummies
            'temp_avg', 'temp2', 'rainfall', 'rain2',
            'alti', 'alti2', 'landlocked']
}

# construct the ols model tbl
tbl5 = tbl_56_regr(df, groups, regressors, y, country_dummies)

# check correctness
with open("table5_regression_output.txt", "w") as f:
    for label, result in tbl5.items():
        f.write(f"\n===== Table‑5 column {label} =====\n")
        f.write(result.summary().as_text())
        f.write("\n\n")

In [ ]:
def tbl_create(tbl, row_order, row_labels, contrasts):
    """ helper function to tables that matches paper """
    # construct the upper rows
    table_rows = []
    for par in row_order:
        table_rows.append(
            [row_labels[par]] +
            [coef_cell(tbl[f"({i})"], par) for i in range(1, len(tbl)+1)]
        )
    table_rows.append(
        ["Observations"] + [f"{int(tbl[f'({i})'].nobs):.0f}" for i in range(1, len(tbl)+1)]
    )
    table_rows.append(
        ["$R^2$"] + [f"{tbl[f'({i})'].rsquared:.3f}" for i in range(1, len(tbl)+1)]
    )

    # construct lower rows
    contrast_rows = []
    for label, (num, den) in contrasts.items():
        row = [label]
        for i in range(1, len(tbl)+1):
            res = tbl[f"({i})"]
            # only compute if both params exist in this regression
            if num in res.params.index and den in res.params.index:
                R = np.zeros(len(res.params))
                R[ res.params.index.get_loc(num) ] =  1
                R[ res.params.index.get_loc(den) ] = -1
                test = res.t_test(R)
                if "Coefficient" in label:               # difference of coefs
                    cell = f"{test.effect[0]:.3f}"
                else:        
                    test = res.f_test(R)
                    cell = f"({test.pvalue:.3f})"
            else:
                cell = ""
            row.append(cell)
        contrast_rows.append(row)
    
    # merge upper and lower rows
    for r in contrast_rows:
        table_rows.append(r)
    
    return table_rows
    
    

In [ ]:

row_order = [
    "good", "bad", "ugly",
    "lpopd",
    "plantations", "mining",
    "temp_avg", "temp2",
    "rainfall", "rain2",
    "alti", "alti2",
    "landlocked"
]

row_labels = {
    "good"      : "Good activities dummy",
    "bad"       : "Bad activities dummy",
    "ugly"      : "Ugly activities dummy",
    "lpopd"     : "Log precolonial population density",
    "plantations":"Plantations dummy",
    "mining"    : "Mining dummy",
    "temp_avg"  : "Average temperature",
    "temp2"     : "Average temperature squared",
    "rainfall"  : "Total rainfall",
    "rain2"     : "Total rainfall squared",
    "alti"      : "Altitude (per km)",
    "alti2"     : "Altitude squared",
    "landlocked": "Landlocked dummy"
}

contrasts = {
    "Coefficient Bad − Good"              : ("bad", "good"),
    "$F$‑test: Good = Bad $p$‑value"       : ("good", "bad"),
    "Coefficient Ugly − Good"             : ("ugly", "good"),
    "$F$‑test: Good = Ugly $p$‑value"      : ("good", "ugly"),
    "Coefficient Bad − Ugly"              : ("bad", "ugly"),
    "$F$‑test: Bad = Ugly $p$‑value"       : ("bad", "ugly"),
    "Coefficient Plantations − Mining"    : ("plantations", "mining"),
    "$F$‑test: Plantations = Mining $p$‑value": ("plantations", "mining"),
}

tbl5_df = pd.DataFrame(tbl_create(tbl5, row_order, row_labels, contrasts))
tbl5_df

,0,1,2,3,4,5,6
0,Good activities dummy,-0.061\n(0.101),,-0.019\n(0.091),0.018\n(0.081),0.001\n(0.076),0.004\n(0.076)
1,Bad activities dummy,-0.392***\n(0.099),,-0.286***\n(0.088),-0.293***\n(0.085),-0.276***\n(0.081),
2,Ugly activities dummy,-0.263***\n(0.099),,-0.123\n(0.090),-0.138*\n(0.083),-0.159**\n(0.078),-0.166**\n(0.078)
3,Log precolonial population density,,-0.078***\n(0.024),-0.059**\n(0.024),-0.056**\n(0.022),-0.052***\n(0.020),-0.053***\n(0.020)
4,Plantations dummy,,,,,,-0.231**\n(0.100)
5,Mining dummy,,,,,,-0.318***\n(0.096)
6,Average temperature,,,,0.002\n(0.016),-0.002\n(0.015),-0.002\n(0.015)
7,Average temperature squared,,,,-0.000\n(0.000),-0.001\n(0.000),-0.001\n(0.000)
8,Total rainfall,,,,-0.211**\n(0.083),-0.198**\n(0.081),-0.207**\n(0.082)
9,Total rainfall squared,,,,0.026\n(0.020),0.022\n(0.020),0.024\n(0.020)


In [123]:
tbl5_df.to_latex("table5.tex", escape=False)

In [ ]:
""" Replicate Table 6. Colonial Activities and Current Poverty Rate"""
# data cleaning, drop out regions with no poverty rate
df = colo_regr_df.copy()
df = df.loc[df['lpoverty'].notna()].reset_index(drop=True)
y  = df['lpoverty']
groups = df['lpopd']
country_dummies = pd.get_dummies(df['country'], drop_first=True, dtype=float)

regressors = {
    '(1)': ['good', 'bad', 'ugly'],
    '(2)': ['lpopd'],
    '(3)': ['good', 'bad', 'ugly', 'lpopd'],
    '(4)': ['good', 'bad', 'ugly', 'lpopd',
            'temp_avg', 'temp2', 'rainfall', 'rain2'],
    '(5)': ['good', 'bad', 'ugly', 'lpopd',
            'temp_avg', 'temp2', 'rainfall', 'rain2',
            'alti', 'alti2', 'landlocked']
}

# do regression and get ols models
tbl6 = tbl_56_regr(df, groups, regressors, y, country_dummies)

# build final table
row_order = [
    "good", "bad", "ugly",
    "lpopd",
    "temp_avg", "temp2",
    "rainfall", "rain2",
    "alti", "alti2",
    "landlocked"
]
contrasts = {
    "Coefficient Bad − Good"              : ("bad", "good"),
    "$F$‑test: Good = Bad $p$‑value"       : ("good", "bad"),
    "Coefficient Ugly − Good"             : ("ugly", "good"),
    "$F$‑test: Good = Ugly $p$‑value"      : ("good", "ugly"),
    "Coefficient Bad − Ugly"              : ("bad", "ugly"),
    "$F$‑test: Bad = Ugly $p$‑value"       : ("bad", "ugly")
}
tbl6_df = pd.DataFrame(tbl_create(tbl6, row_order, row_labels, contrasts))
tbl6_df

,0,1,2,3,4,5
0,Good activities dummy,0.091\n(0.108),,0.054\n(0.096),0.001\n(0.081),0.035\n(0.071)
1,Bad activities dummy,0.297***\n(0.110),,0.202**\n(0.098),0.194**\n(0.090),0.164*\n(0.094)
2,Ugly activities dummy,0.095\n(0.118),,-0.029\n(0.107),-0.026\n(0.098),0.001\n(0.104)
3,Log precolonial population density,,0.054**\n(0.027),0.053**\n(0.027),0.052**\n(0.025),0.046**\n(0.023)
4,Average temperature,,,,-0.000\n(0.038),0.015\n(0.030)
5,Average temperature squared,,,,0.000\n(0.001),0.001\n(0.001)
6,Total rainfall,,,,0.267***\n(0.081),0.252***\n(0.078)
7,Total rainfall squared,,,,-0.028*\n(0.015),-0.023\n(0.014)
8,Altitude (per km),,,,,0.028\n(0.119)
9,Altitude squared,,,,,0.059*\n(0.034)


In [125]:
tbl6_df.to_latex("table6.tex", escape=False)

In [ ]:
def tbl_9_regr(df, columns):
    """ helper function to do the regression for table 9 panel A"""
    tbl = {} # dictionary {og-column‑label : fitted result}

    for og_col, col_name in columns.items():
        reg_df = df.loc[df[og_col].notna()].reset_index(drop=True)
        # make sure to include country dummies
        X_base = ['good', 'bad', 'ugly', 'lpopd',
            'temp_avg', 'temp2', 'rainfall', 'rain2',
            'alti', 'alti2', 'landlocked']
        country_dummies = pd.get_dummies(reg_df['country'], drop_first=True, dtype=float)
        groups = reg_df['lpopd']
        y = reg_df[og_col]
        X = sm.add_constant(pd.concat([reg_df[X_base], country_dummies], axis=1))
        res = sm.OLS(y, X).fit(
                cov_type = 'cluster',
                cov_kwds = {'groups': groups}
            )
        tbl[col_name] = res
    return tbl


In [ ]:
""" Replicate Table 9.Panel A"""
columns = {
    'lyppp': '(1)',
    'lgini': '(2)',
    'lschoolspk': '(3)',
    'llit': '(4)',
    'lseats': '(5)'
}
tbl9_panelA = tbl_9_regr(colo_regr_df, columns)

# check correctness
with open("table9_regression_output.txt", "w") as f:
    for label, result in tbl9_panelA.items():
        f.write(f"\n===== Table‑9 column {label} =====\n")
        f.write(result.summary().as_text())
        f.write("\n\n")

In [ ]:
row_order = [
    "good", "bad", "ugly",
    "lpopd"
]
row_labels = {
    "good"      : "Good activities dummy",
    "bad"       : "Bad activities dummy",
    "ugly"      : "Ugly activities dummy",
    "lpopd"     : "Log precolonial population density"
}
contrasts = {
    "Coefficient Bad − Good"              : ("bad", "good"),
    "$F$‑test: Good = Bad $p$‑value"       : ("good", "bad"),
    "Coefficient Ugly − Good"             : ("ugly", "good"),
    "$F$‑test: Good = Ugly $p$‑value"      : ("good", "ugly"),
    "Coefficient Bad − Ugly"              : ("bad", "ugly"),
    "$F$‑test: Bad = Ugly $p$‑value"       : ("bad", "ugly")
}

tbl9_panelA_df = pd.DataFrame(tbl_create(tbl9_panelA, row_order, row_labels, contrasts))
tbl9_panelA_df = tbl9_panelA_df.rename(
    columns={1: 'Log GDP per Capita\n(1)',
    2: 'Log Gini Index\n(2)',
    3: 'Log Schools per Child\n(3)',
    4: 'Log Literacy Rate\n(4)',
    5: 'Log Seats in Lower House per Voter\n(5)'}
    )
tbl9_panelA_df


,0,Log GDP per Capita\n(1),Log Gini Index\n(2),Log Schools per Child\n(3),Log Literacy Rate\n(4),Log Seats in Lower House per Voter\n(5)
0,Good activities dummy,0.001\n(0.076),0.001\n(0.014),0.073\n(0.073),-0.006\n(0.010),-0.014\n(0.089)
1,Bad activities dummy,-0.276***\n(0.081),0.017\n(0.018),-0.011\n(0.085),-0.023\n(0.015),-0.273**\n(0.114)
2,Ugly activities dummy,-0.159**\n(0.078),-0.005\n(0.020),-0.075\n(0.094),-0.016\n(0.016),-0.321**\n(0.129)
3,Log precolonial population density,-0.052***\n(0.020),0.000\n(0.007),-0.014\n(0.018),-0.001\n(0.003),-0.070**\n(0.028)
4,Observations,345,268,317,270,318
5,$R^2$,0.829,0.738,0.713,0.646,0.861
6,Coefficient Bad − Good,-0.277,0.016,-0.084,-0.017,-0.259
7,$F$‑test: Good = Bad $p$‑value,(0.004),(0.354),(0.160),(0.356),(0.022)
8,Coefficient Ugly − Good,-0.160,-0.006,-0.148,-0.010,-0.307
9,$F$‑test: Good = Ugly $p$‑value,(0.080),(0.761),(0.033),(0.568),(0.021)


In [158]:
tbl9_panelA_df.to_latex("table9_panelA.tex", escape=False)